# 0027 / 05 Dual-T4 canonical BC

Train the complete 0025 CanonicalSemanticPolicy architecture from random initialization on all four winner-only daily datasets. No W&B run or pretrained checkpoint is used.


In [ ]:
from __future__ import annotations
import gzip, importlib, json, math, random, shutil, sys, time, zipfile
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, IterableDataset, get_worker_info

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("0027 requires Kaggle NvidiaTeslaT4 x2")
print({"cuda_count": torch.cuda.device_count(), "devices": [torch.cuda.get_device_name(i) for i in range(2)]})

SEED = 20260803
ROLLING_CYCLES = 1
EXPECTED_PARTS = {
    "01_days_0715_0719": tuple(f"2026-07-{day:02d}" for day in range(15, 20)),
    "02_days_0720_0724": tuple(f"2026-07-{day:02d}" for day in range(20, 25)),
    "03_days_0725_0728": tuple(f"2026-07-{day:02d}" for day in range(25, 29)),
    "04_days_0729_0801": ("2026-07-29", "2026-07-30", "2026-07-31", "2026-08-01"),
}
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

input_root = Path("/kaggle/input")
expected_dates = {date for dates in EXPECTED_PARTS.values() for date in dates}
manifest_candidates = {}
for path in input_root.rglob("manifest.json"):
    try:
        candidate = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    date = candidate.get("date")
    counts = candidate.get("split_counts", {})
    if (candidate.get("schema_version") == "0025_canonical_semantic_decision_v2"
            and candidate.get("partition_schema") == "0027_daily_semantic_dataset_v1"
            and candidate.get("status") == "complete"
            and candidate.get("winner_only") is True
            and date in expected_dates
            and int(counts.get("train", 0)) > 0
            and int(counts.get("validation", 0)) > 0
            and set(candidate.get("shards", {})) >= {"train", "validation"}):
        if date in manifest_candidates:
            raise RuntimeError(f"duplicate attached daily manifest: {date}")
        manifest_candidates[date] = (path, candidate)
if set(manifest_candidates) != expected_dates:
    missing = sorted(expected_dates - set(manifest_candidates))
    raise FileNotFoundError(f"expected 18 complete non-empty daily manifests; missing={missing}")
def source_candidates(root: Path) -> list[Path]:
    return sorted({
        path.parent.parent
        for path in root.rglob("official_public_prototypes_v1.json")
        if path.parent.name == "assets" and (path.parent.parent / "model" / "canonical").is_dir()
    })

def find_source_root(input_root: Path) -> Path:
    candidates = source_candidates(input_root)
    if len(candidates) == 1:
        return candidates[0]
    extract_root = Path("/kaggle/working/ptcg_0027_0025_source")
    for archive in sorted(input_root.rglob("0025_semantic_foundation_pretraining.zip")):
        with zipfile.ZipFile(archive) as bundle:
            if any(name.endswith("assets/official_public_prototypes_v1.json") for name in bundle.namelist()):
                bundle.extractall(extract_root)
                candidates = source_candidates(extract_root)
                if len(candidates) == 1:
                    return candidates[0]
    raise FileNotFoundError("attach the vendored 0025 source package or 0025_semantic_foundation_pretraining.zip")

source_root = find_source_root(input_root)
sys.path.insert(0, str(source_root.parent))
package = source_root.name
PrototypeIndex = importlib.import_module(f"{package}.features.prototypes").PrototypeIndex
collate_canonical_records = importlib.import_module(f"{package}.features.canonical.batching").collate_canonical_records
ACTOR_KEYS = importlib.import_module(f"{package}.features.canonical.schema").ACTOR_KEYS
canonical_model = importlib.import_module(f"{package}.model.canonical")
CanonicalModelConfig, CanonicalSemanticPolicy = canonical_model.CanonicalModelConfig, canonical_model.CanonicalSemanticPolicy
if len(ACTOR_KEYS) != 22:
    raise RuntimeError(f"expected all 22 canonical actor features, got {len(ACTOR_KEYS)}")

class Rows(IterableDataset):
    def __init__(self, paths, seed=SEED):
        self.paths = tuple(paths)
        self.seed = seed

    def __iter__(self):
        worker = get_worker_info()
        worker_id = 0 if worker is None else worker.id
        worker_count = 1 if worker is None else worker.num_workers
        paths = list(self.paths)
        random.Random(self.seed).shuffle(paths)
        paths = paths[worker_id::worker_count]
        for path in paths:
            with gzip.open(path, "rt", encoding="utf-8") as handle:
                for line in handle:
                    if line.strip():
                        yield json.loads(line)

paths_by_part = {part: {"train": [], "validation": []} for part in EXPECTED_PARTS}
inventory = []
for part, dates in EXPECTED_PARTS.items():
    for date in dates:
        manifest_path, manifest = manifest_candidates[date]
        for split in ("train", "validation"):
            shards = manifest["shards"][split]
            if not shards:
                raise RuntimeError(f"{date} has no {split} shards")
            for item in shards:
                shard = manifest_path.parent / item["path"]
                if not shard.is_file() or shard.stat().st_size <= 0:
                    raise FileNotFoundError(f"missing or empty shard: {shard}")
                paths_by_part[part][split].append(shard)
        inventory.append({"part": part, "date": date, "records": manifest["records"], "episodes": manifest["episodes"], "split_counts": manifest["split_counts"]})

first_shard = paths_by_part[next(iter(EXPECTED_PARTS))]["train"][0]
with gzip.open(first_shard, "rt", encoding="utf-8") as handle:
    first_record = json.loads(next(line for line in handle if line.strip()))
if set(first_record.get("actor", {})) != ACTOR_KEYS:
    raise RuntimeError("dataset record does not contain the complete canonical actor schema")
prototype_path = source_root / "assets" / "official_public_prototypes_v1.json"
prototypes = PrototypeIndex.load(prototype_path)
model_config = CanonicalModelConfig()
policy = CanonicalSemanticPolicy(model_config, prototypes).to("cuda:0")
parameter_count = sum(parameter.numel() for parameter in policy.parameters())
print(json.dumps({"event": "preflight", "dates": sorted(expected_dates), "train_records": sum(row["split_counts"]["train"] for row in inventory), "validation_records": sum(row["split_counts"]["validation"] for row in inventory), "winner_only": True, "actor_feature_count": len(ACTOR_KEYS), "actor_features": sorted(ACTOR_KEYS), "model_config": model_config.to_dict(), "parameter_count": parameter_count, "initialized_from_checkpoint": None}), flush=True)

class TeacherLogits(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, batch):
        return self.policy.teacher_logits(batch)

distributed = nn.DataParallel(TeacherLogits(policy), device_ids=[0, 1])
optimizer = torch.optim.AdamW(distributed.parameters(), lr=3e-4, weight_decay=0.02)
scaler = torch.amp.GradScaler("cuda", enabled=True, init_scale=4096.0, growth_interval=10000)
amp_dtype = torch.float16  # Nvidia T4 Tensor Cores support FP16, not BF16.

def loader(paths, batch_size, seed=SEED):
    return DataLoader(Rows(paths, seed=seed), batch_size=batch_size, num_workers=2, pin_memory=True, collate_fn=collate_canonical_records)

def move(batch):
    return {key: value.to("cuda:0", non_blocking=True) for key, value in batch.items()}

smoke_batch = move(collate_canonical_records([first_record]))
distributed.eval()
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=amp_dtype):
    smoke_logits = distributed(smoke_batch)
if smoke_logits.ndim != 3 or not torch.isfinite(smoke_logits).all():
    raise RuntimeError("full canonical model forward preflight failed")
del smoke_batch, smoke_logits
torch.cuda.empty_cache()

out = Path("/kaggle/working/ptcg_0027_dual_t4_train")
out.mkdir(parents=True, exist_ok=True)
history = []
best = None
best_validation_loss = math.inf
parts = list(EXPECTED_PARTS)
total_rolling_epochs = ROLLING_CYCLES * len(parts)
started = time.time()
for epoch in range(1, total_rolling_epochs + 1):
    part = parts[(epoch - 1) % len(parts)]
    cycle = (epoch - 1) // len(parts) + 1
    train_loader = loader(paths_by_part[part]["train"], 192, SEED + epoch)
    distributed.train()
    train_loss_sum, train_tokens, train_correct, train_decisions, seen_decisions = 0.0, 0, 0, 0, 0
    skipped_updates = 0
    for batch_index, raw in enumerate(train_loader, 1):
        batch = move(raw)
        seen_decisions += int(batch["targets"].size(0))
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits = distributed(batch)
            mask = batch["targets"].ne(-100)
            loss = F.cross_entropy(logits[mask], batch["targets"][mask])
        if not torch.isfinite(loss):
            raise FloatingPointError("non-finite BC loss")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = nn.utils.clip_grad_norm_(distributed.parameters(), 1.0)
        if not torch.isfinite(grad_norm):
            skipped_updates += 1
            optimizer.zero_grad(set_to_none=True)
            scaler.update()
            print(json.dumps({"event": "amp_update_skipped", "rolling_epoch": epoch, "partition": part, "batch": batch_index, "skipped_updates": skipped_updates, "new_scale": scaler.get_scale()}), flush=True)
            continue
        scaler.step(optimizer)
        scaler.update()
        token_count = int(mask.sum().item())
        train_loss_sum += float(loss.detach().cpu()) * token_count
        train_tokens += token_count
        train_correct += int((logits.argmax(-1).eq(batch["targets"]) & mask).sum().item())
        train_decisions += int(batch["targets"].size(0))
        if batch_index == 1 or batch_index % 100 == 0:
            print(json.dumps({"event": "train_progress", "rolling_epoch": epoch, "partition": part, "batch": batch_index, "loss": train_loss_sum / max(train_tokens, 1), "decisions": train_decisions}), flush=True)
    if not train_tokens:
        raise RuntimeError(f"rolling epoch has no training tokens: {part}")
    expected_train_decisions = sum(row["split_counts"]["train"] for row in inventory if row["part"] == part)
    if seen_decisions != expected_train_decisions:
        raise RuntimeError(f"training loader coverage mismatch for {part}: expected={expected_train_decisions} seen={seen_decisions}")
    validation = None
    if epoch % len(parts) == 0:
        valid_paths = [path for name in parts for path in paths_by_part[name]["validation"]]
        valid_loader = loader(valid_paths, 256, SEED)
        distributed.eval()
        valid_loss_sum, valid_tokens, valid_correct, valid_exact, valid_decisions = 0.0, 0, 0, 0, 0
        with torch.inference_mode():
            for raw in valid_loader:
                batch = move(raw)
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    logits = distributed(batch)
                    mask = batch["targets"].ne(-100)
                    loss_sum = F.cross_entropy(logits[mask], batch["targets"][mask], reduction="sum")
                matches = logits.argmax(-1).eq(batch["targets"]) & mask
                valid_loss_sum += float(loss_sum.cpu())
                valid_tokens += int(mask.sum().item())
                valid_correct += int(matches.sum().item())
                valid_exact += int((matches | ~mask).all(1).sum().item())
                valid_decisions += int(batch["targets"].size(0))
        if not valid_tokens or not valid_decisions:
            raise RuntimeError("all-part validation is empty")
        expected_valid_decisions = sum(row["split_counts"]["validation"] for row in inventory)
        if valid_decisions != expected_valid_decisions:
            raise RuntimeError(f"validation loader coverage mismatch: expected={expected_valid_decisions} seen={valid_decisions}")
        validation = {"loss": valid_loss_sum / valid_tokens, "token_accuracy": valid_correct / valid_tokens, "teacher_exact_action": valid_exact / valid_decisions, "tokens": valid_tokens, "decisions": valid_decisions}
    row = {"rolling_epoch": epoch, "cycle": cycle, "partition": part, "train": {"loss": train_loss_sum / train_tokens, "token_accuracy": train_correct / train_tokens, "tokens": train_tokens, "decisions": train_decisions, "seen_decisions": seen_decisions, "expected_decisions": expected_train_decisions, "skipped_updates": skipped_updates}, "validation_all_parts": validation, "elapsed_seconds": time.time() - started}
    history.append(row)
    payload = {"schema_version": "0027_canonical_bc_model_v1", "architecture": "CanonicalSemanticPolicy", "state_dict": {key: value.detach().cpu() for key, value in distributed.module.policy.state_dict().items()}, "model_config": model_config.to_dict(), "rolling_epoch": epoch, "validation": validation, "initialized_from_checkpoint": None, "winner_only": True, "actor_features": sorted(ACTOR_KEYS)}
    torch.save(payload, out / "last_model.pt")
    if validation is not None and validation["loss"] < best_validation_loss:
        best_validation_loss = validation["loss"]
        best = row
        shutil.copy2(out / "last_model.pt", out / "best_model.pt")
    report = {"schema_version": "0027_canonical_bc_training_report_v1", "state": "training", "experiment": "0027_semantic_foundation_pretraining", "architecture": "CanonicalSemanticPolicy", "model_config": model_config.to_dict(), "parameter_count": parameter_count, "initialized_from_checkpoint": None, "wandb": None, "winner_only": True, "actor_feature_count": len(ACTOR_KEYS), "actor_features": sorted(ACTOR_KEYS), "gpu_count": 2, "rolling_cycles": ROLLING_CYCLES, "partitions": parts, "dataset_inventory": inventory, "history": history, "best": best}
    (out / "training_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(json.dumps({"event": "rolling_epoch_complete", **row}), flush=True)
if best is None:
    raise RuntimeError("training completed without an all-part validation model")
report["state"] = "complete"
report["elapsed_seconds"] = time.time() - started
(out / "training_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
shutil.copy2(prototype_path, out / prototype_path.name)
(out / "model_contract.json").write_text(json.dumps({"schema_version": "0027_canonical_model_contract_v1", "architecture": "CanonicalSemanticPolicy", "model_config": model_config.to_dict(), "actor_view": "all_22_canonical_features", "actor_features": sorted(ACTOR_KEYS), "action_contract": "ordered_legal_option_pointer_plus_stop", "initialized_from_checkpoint": None, "winner_only": True}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps({"event": "training_complete", "output": str(out), "best": best}), flush=True)

